# Konsolidierung der PRC-Ist-Datenbasis

Dieses Notebook baut aus den PRC-Telematikdateien eine bereinigte Ist-Basis und speichert sie als CSV und Pickle. Es folgt demselben Rückgrat wie Soll und EMR: einlesen, auf März filtern, Mafi entfernen, Kennzeichen normalisieren, Dubletten entfernen, exportieren. Den größten Unterschied macht der Anfang: Die PRC-Daten sind heterogenes XML aus tausenden Einzeldateien, deren Struktur erst gesichtet und über einen Parser eingelesen werden muss, bevor die eigentliche Bereinigung beginnt. Die Daten liegen am Ende auf Meldungsebene vor, eine Tour-Zuordnung entsteht erst im Analyse-Notebook.

## 1. Struktur der PRC-Dateien sichten

Die PRC-Dateien sind heterogenes XML: eine einzelne Datei kann mehrere Meldungen unterschiedlichen Typs enthalten. Welche Typen es gibt, steht in der C-Logistic-Schnittstellenbeschreibung: Position, FMSStatus, Tourstatus, Stationsstatus und Sendposstatus. Bevor die Daten eingelesen werden, geben wir der Übersicht halber je ein vollständiges Beispiel pro Typ aus, damit wir sehen, wie die Meldungen aufgebaut sind und welche Felder wir später übernehmen.

In [1]:
import re
import pandas as pd
from pathlib import Path

# Anzeigeoptionen, damit DataFrames vollständig sichtbar sind
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", None)

BASE_PATH = Path.cwd().parent / "data" / "raw"

prc_files = sorted(BASE_PATH.glob("msg_*.prc"))
print(f"Gefundene PRC-Dateien: {len(prc_files)}")

# Die fünf Meldungstypen laut C-Logistic-Schnittstellenbeschreibung
bekannte_typen = ["Sendposstatus", "Stationsstatus", "FMSStatus", "Tourstatus", "Position"]

# Pro Typ das erste Vorkommen festhalten und abbrechen, sobald alle fünf gesehen sind
# (früher Abbruch, damit nicht alle 24.521 Dateien gelesen werden)
beispiele = {}
for pfad in prc_files:
    inhalt = pfad.read_text(encoding="utf-8")
    for typ in bekannte_typen:
        if typ not in beispiele and f"<{typ}>" in inhalt:
            beispiele[typ] = (pfad.name, inhalt)
    if len(beispiele) == len(bekannte_typen):
        break

# Je Typ ein vollständiges Beispiel ausgeben, um den Aufbau zu sehen
print(f"Beispiele für die {len(bekannte_typen)} Meldungstypen:")
for typ, (name, inhalt) in beispiele.items():
    print(f"Meldungstyp: {typ}    (Datei: {name})")
    print(inhalt)

Gefundene PRC-Dateien: 24521
Beispiele für die 5 Meldungstypen:
Meldungstyp: Position    (Datei: msg_20260228000043_956.imp.20260228000310889.prc)
﻿<Import>
  <Position>
    <GeoPosition>
      <PositionsID>16049334470</PositionsID>
      <Longitude>9.98548</Longitude>
      <Latitude>53.52335</Latitude>
      <Zeitstempel_GPS>2026-02-27T23:59:59+01:00</Zeitstempel_GPS>
      <Richtung>0</Richtung>
      <Geschwindigkeit>0</Geschwindigkeit>
      <KMStand>339687</KMStand>
    </GeoPosition>
    <Zeitstempel_Fahrzeug>2026-02-28T00:00:00+01:00</Zeitstempel_Fahrzeug>
    <Zeitstempel_Server>2026-02-28T00:00:43</Zeitstempel_Server>
    <FahrzeugID>Plö-TS856</FahrzeugID>
  </Position>
</Import>
Meldungstyp: FMSStatus    (Datei: msg_20260301141707_983.imp.20260301141842821.prc)
﻿<Import>
  <Tourstatus>
    <GeoPosition>
      <PositionsID>16053897222</PositionsID>
      <Longitude>9.98927</Longitude>
      <Latitude>53.52262</Latitude>
      <Zeitstempel_GPS>2026-03-01T14:13:25+01:00</Zeitst

Bevor wir einlesen, halten wir fest, wie sich die Typen unterscheiden und welchen Mehrwert sie für das weitere Vorgehen haben könnten, aus den Beispielen oben und aus der C-Logistic-Beschreibung:

- **Position** nur GPS (Ort, Geschwindigkeit, Kilometerstand) mit Zeitstempel und Fahrzeug, kein Status, kein Tour- oder Stationsbezug. Die lückenlose Positionsspur und damit potenzielle Grundlage für Geofencing, Route und Standzeit.
- **Stationsstatus** trägt eine StationID und einen Status und bildet den Stations-Zyklus einer Tour ab, von der Anfahrt bis zur Abfahrt. Dürfte die zentrale Quelle für Standzeiten und die tatsächliche Stopp-Reihenfolge sein.
- **Sendposstatus** trägt eine SendposID und einen Status und bezieht sich auf eine einzelne Sendungsposition. Könnte zeigen, was an einem Stopp tatsächlich geladen oder entladen wurde.
- **Tourstatus** trägt eine TourID und einen Status auf Tour-Ebene und klammert eine Tour zeitlich.
- **FMSStatus** scheinen Fahrzeugsignale der Flotten-Management-Schnittstelle zu sein, in Werten, ohne Status und ohne Tour- oder Stationsbezug. 

## 2. Einlesen und Zusammenführen der PRC-Meldungen

Jede Datei hat ein Import-Wurzelelement, dessen direkte Kinder die einzelnen Meldungen sind, und eine Datei kann mehrere enthalten. Wir erzeugen daher eine Zeile je Kind und übernehmen den Tag-Namen als msg_typ. Pro Meldung behalten wir die Felder, die für den Soll-Ist-Vergleich und die Routen- und Standzeitanalyse infrage kommen: aus der GeoPosition Längen- und Breitengrad, Geschwindigkeit, Kilometerstand und den Zeitstempel_GPS, auf Meldungsebene das Fahrzeug, die Zeitstempel von Fahrzeug und Server, den Status sowie die Schlüssel TourID, StationID und SendposID. Die Richtung lassen wir weg, da sich die Fahrtrichtung aus der Abfolge der Positionen ergibt.

Den Werte-Block lesen wir zunächst nur dort mit, wo er überhaupt auftaucht. Welche Typen Werte tragen und ob wir sie brauchen, lässt sich erst nach dem Einlesen beurteilen, deshalb sammeln wir sie vorerst und sehen sie uns später an. Welcher der beiden Zeitstempel sich als Meldungszeitpunkt eignet, klären wir ebenfalls erst, wenn die Zeitbereiche vorliegen.

In [2]:
import xml.etree.ElementTree as ET

# Felder unter <GeoPosition>
GEO_FELDER = ["PositionsID", "Longitude", "Latitude", "Geschwindigkeit", "KMStand", "Zeitstempel_GPS"]
# Direkte Felder unter dem Meldungselement
MELDUNG_FELDER = ["Zeitstempel_Fahrzeug", "Zeitstempel_Server", "FahrzeugID",
                  "Status", "TourID", "StationID", "SendposID"]

def parse_prc_datei(pfad):
    """Eine Zeile je Meldungselement unter <Import>. FMSStatus trägt zusätzlich
    mehrere <Werte> (Typ/Wert), die separat gesammelt werden."""
    zeilen, fms_werte = [], []
    root = ET.parse(pfad).getroot()
    for meldung in root:
        eintrag = {"msg_typ": meldung.tag, "quelle_datei": pfad.name}
        geo = meldung.find("GeoPosition")
        for feld in GEO_FELDER:
            eintrag[feld] = geo.findtext(feld)        # findtext gibt None, falls das Feld fehlt
        for feld in MELDUNG_FELDER:
            eintrag[feld] = meldung.findtext(feld)     # so bleiben typfremde Felder einfach leer
        zeilen.append(eintrag)
        if meldung.tag == "FMSStatus":                 # nur FMSStatus trägt einen Werte-Block
            for w in meldung.findall("Werte"):
                fms_werte.append({"quelle_datei": pfad.name,
                                  "PositionsID": geo.findtext("PositionsID"),
                                  "Typ": w.findtext("Typ"), "Wert": w.findtext("Wert")})
    return zeilen, fms_werte

alle_zeilen, alle_fms = [], []
for pfad in prc_files:
    zeilen, fms = parse_prc_datei(pfad)
    alle_zeilen.extend(zeilen)
    alle_fms.extend(fms)

prc_raw = pd.DataFrame(alle_zeilen)
fms_werte_raw = pd.DataFrame(alle_fms)

print(f"Verarbeitete Dateien: {len(prc_files)}")
print(f"Erzeugte Meldungszeilen: {len(prc_raw)}")
print(f"FMS-Werte-Einträge: {len(fms_werte_raw)}")
print("Verteilung der Meldungstypen:")
print(prc_raw["msg_typ"].value_counts(dropna=False))
prc_raw.head()

Verarbeitete Dateien: 24521
Erzeugte Meldungszeilen: 33155
FMS-Werte-Einträge: 43170
Verteilung der Meldungstypen:
msg_typ
Position          11535
FMSStatus          8634
Stationsstatus     8299
Sendposstatus      3280
Tourstatus         1405
Ereignis              2
Name: count, dtype: int64


,msg_typ,quelle_datei,PositionsID,Longitude,Latitude,Geschwindigkeit,KMStand,Zeitstempel_GPS,Zeitstempel_Fahrzeug,Zeitstempel_Server,FahrzeugID,Status,TourID,StationID,SendposID
0,Position,msg_20260228000043_956.imp.20260228000310889.prc,16049334470,9.98548,53.52335,0,339687,2026-02-27T23:59:59+01:00,2026-02-28T00:00:00+01:00,2026-02-28T00:00:43,Plö-TS856,NaN,NaN,NaN,NaN
1,Position,msg_20260228000043_957.imp.20260228000310923.prc,16049339872,9.98613,53.5234,0,502579,2026-02-27T23:59:59+01:00,2026-02-28T00:00:00+01:00,2026-02-28T00:00:43,TS859,NaN,NaN,NaN,NaN
2,Position,msg_20260228000315_958.imp.20260228000815933.prc,16049349544,9.98619,53.52356,0,183989,2026-02-27T20:53:12+01:00,2026-02-28T00:02:42+01:00,2026-02-28T00:03:15,OD-TS 8050,NaN,NaN,NaN,NaN
3,Position,msg_20260228000345_959.imp.20260228000815958.prc,16049351179,9.98592,53.52345,0,461200,2026-02-27T23:59:59+01:00,2026-02-28T00:00:00+01:00,2026-02-28T00:03:45,OD-TS 8046,NaN,NaN,NaN,NaN
4,Position,msg_20260228060024_960.imp.20260228060348749.prc,16050228689,9.98616,53.52342,0,502579,2026-02-28T05:59:59+01:00,2026-02-28T06:00:00+01:00,2026-02-28T06:00:24,TS859,NaN,NaN,NaN,NaN


### 2.1 Befüllung je Meldungstyp prüfen

Der Parser fragt für jede Meldung alle Felder ab, unabhängig vom Typ. Wir prüfen daher, welche Felder tatsächlich gefüllt sind, aufgeschlüsselt nach Meldungstyp. Das bestätigt zweierlei auf einmal: Die GeoPosition-Felder (Koordinaten, Kilometerstand, GPS-Zeit) sind über alle Typen zu 100 Prozent gefüllt, und die Schlüssel TourID, StationID und SendposID kommen jeweils nur bei ihrem eigenen Meldungstyp vor. Damit ist die Annahme aus dem Parser, dass jede Meldung eine vollständige GeoPosition trägt, belegt — und das typgebundene Befüllungsmuster sichtbar.

In [3]:
# Befüllung je Spalte nach Meldungstyp
befuellung = prc_raw.groupby("msg_typ").apply(
    lambda g: (g.notna().mean() * 100).round(0), include_groups=False
)
befuellung[["Longitude", "Latitude", "KMStand", "Zeitstempel_GPS",
            "Status", "TourID", "StationID", "SendposID"]]

,Longitude,Latitude,KMStand,Zeitstempel_GPS,Status,TourID,StationID,SendposID
msg_typ,,,,,,,,
Ereignis,100.0,100.0,100.0,100.0,0.0,0.0,0.0,0.0
FMSStatus,100.0,100.0,100.0,100.0,0.0,0.0,0.0,0.0
Position,100.0,100.0,100.0,100.0,0.0,0.0,0.0,0.0
Sendposstatus,100.0,100.0,100.0,100.0,100.0,0.0,0.0,100.0
Stationsstatus,100.0,100.0,100.0,100.0,100.0,0.0,100.0,0.0
Tourstatus,100.0,100.0,100.0,100.0,100.0,100.0,0.0,0.0


### 2.2 Der sechste Meldungstyp Ereignis

Die Typverteilung zeigt neben den fünf dokumentierten Typen einen sechsten, Ereignis, mit nur zwei Vorkommen im ganzen Monat. Das flache Parsing-Schema hat dafür nur die gemeinsamen Felder erfasst. Um zu verstehen, was diese Meldungen tragen, lesen wir die beiden rohen XML-Dateien direkt aus.

In [4]:
ereignis = prc_raw[prc_raw["msg_typ"] == "Ereignis"]
display(ereignis)

# Alle Dateien anzeigen, die <Ereignis> enthalten
for name in ereignis["quelle_datei"].unique():
    print(name)
    print((BASE_PATH / name).read_text(encoding="utf-8-sig"))

,msg_typ,quelle_datei,PositionsID,Longitude,Latitude,Geschwindigkeit,KMStand,Zeitstempel_GPS,Zeitstempel_Fahrzeug,Zeitstempel_Server,FahrzeugID,Status,TourID,StationID,SendposID
4629,Ereignis,msg_20260305081857_4412.imp.20260305082219283.prc,16083444083,9.96948,53.51279,0,461908,2026-03-05T08:15:03+01:00,2026-03-05T08:15:03+01:00,2026-03-05T08:18:57,OD-TS 8046,NaN,NaN,NaN,NaN
21442,Ereignis,msg_20260319132430_16778.imp.20260319132701458.prc,16183750851,9.92304,53.59549,0,807024,2026-03-19T13:24:28+01:00,2026-03-19T13:24:28+01:00,2026-03-19T13:24:30,WL-PL431,NaN,NaN,NaN,NaN


msg_20260305081857_4412.imp.20260305082219283.prc
<Import>
  <FMSStatus>
    <GeoPosition>
      <PositionsID>16083444083</PositionsID>
      <Longitude>9.96948</Longitude>
      <Latitude>53.51279</Latitude>
      <Zeitstempel_GPS>2026-03-05T08:15:03+01:00</Zeitstempel_GPS>
      <Richtung>0</Richtung>
      <Geschwindigkeit>0</Geschwindigkeit>
      <KMStand>461908</KMStand>
    </GeoPosition>
    <Zeitstempel_Fahrzeug>2026-03-05T08:15:03+01:00</Zeitstempel_Fahrzeug>
    <Zeitstempel_Server>2026-03-05T08:18:57</Zeitstempel_Server>
    <FahrzeugID>OD-TS 8046</FahrzeugID>
    <FMSStatusID>16083444083</FMSStatusID>
    <Werte>
      <Typ>4</Typ>
      <Wert>0</Wert>
    </Werte>
    <Werte>
      <Typ>1</Typ>
      <Wert>0</Wert>
    </Werte>
    <Werte>
      <Typ>5</Typ>
      <Wert>0</Wert>
    </Werte>
    <Werte>
      <Typ>2</Typ>
      <Wert>461908.531</Wert>
    </Werte>
    <Werte>
      <Typ>4</Typ>
      <Wert>0</Wert>
    </Werte>
  </FMSStatus>
  <Ereignis>
    <GeoPosition

Im rohen XML trägt jede Ereignis-Meldung ein eigenes Feld Typ (in beiden Fällen 3) und eine EreignisID, dazu GeoPosition, Zeitstempel und Fahrzeug — aber keinen Status und keinen Tour-, Stations- oder Sendpos-Schlüssel. In der Mapping-XML ist Typ 3 dem Ereignis Beginn-Stau zugeordnet (Form 1686). Es sind also fahrzeugbezogene Ereignismeldungen ohne Stations- oder Tourbezug, hier jeweils zusammen mit einer FMSStatus-Meldung in derselben Datei. Das erklärt, warum bei diesen Zeilen Status und alle Schlüssel leer sind.

Für die Standzeit- und Routenanalyse sind sie ohne Stations- oder Tourbezug nicht verwertbar, und zwei Vorkommen sind sehr wenig. Wir entfernen sie trotzdem nicht: Stau-Ereignisse könnten für die Erklärung von Verzögerungen nützlich sein, und sie richten keinen Schaden an, weil sie ohne Status später keinen Status-Text bekommen und im Vergleich nicht ins Gewicht fallen. Die ereignisspezifischen Felder Typ und EreignisID erfasst das Schema bewusst nicht, da sie nur diese zwei Zeilen beträfen.

### 2.3 FMSStatus im Kontext und seine Werte

Bevor wir die FMS-Werte selbst ansehen, klären wir, in welchem Zusammenhang FMSStatus-Meldungen auftreten. Da eine PRC-Datei mehrere Meldungen enthalten kann, prüfen wir je Datei, welche Meldungstypen gemeinsam vorliegen und ob ein FMSStatus je allein in einer Datei steht.

In [5]:
# Je Datei die Menge der enthaltenen Meldungstypen; daraus den Kontext der FMSStatus ablesen
typen_je_datei = prc_raw.groupby("quelle_datei")["msg_typ"].agg(set)
mit_fms = typen_je_datei[typen_je_datei.apply(lambda s: "FMSStatus" in s)]

print("FMSStatus allein in der Datei:", mit_fms.apply(lambda s: s == {"FMSStatus"}).sum())
print("Begleittypen in FMSStatus-Dateien:")
pd.Series([t for s in mit_fms for t in s if t != "FMSStatus"]).value_counts()

FMSStatus allein in der Datei: 0
Begleittypen in FMSStatus-Dateien:


Stationsstatus    5679
Sendposstatus     1920
Tourstatus        1033
Ereignis             2
Name: count, dtype: int64

Kein einziger der 8.634 FMSStatus steht allein in seiner Datei; jeder liegt zusammen mit genau einer weiteren Meldung, überwiegend einem Stationsstatus (5.679), seltener einem Sendpos- (1.920) oder Tourstatus (1.033) und in zwei Fällen einer Ereignismeldung. Die Summe der Begleiter entspricht exakt der Zahl der FMSStatus-Dateien, jede trägt also genau einen Begleiter. Da ein FMSStatus nie allein auftritt, ist seine GeoPosition stets derselbe Fix wie der der Begleitmeldung. Das Auslassen des FMS-Blocks aus der Meldungstabelle kostet daher keine eigenständige Positionsinformation — die FMS-Werte selbst sammeln wir getrennt in der zweiten Tabelle.

### 2.4 Die FMS-Werte sichten

Wir untersuchen die FMS-Signale, die wir separat in fms_werte_raw gesammelt haben und schauen, welche Typ-Codes vorkommen und wie ihre Werte verteilt sind.

In [6]:
# FMS-Werte numerisch machen, dann Häufigkeit und Verteilung je Typ-Code ansehen
fms_werte_raw["Wert"] = pd.to_numeric(fms_werte_raw["Wert"], errors="coerce")

display(fms_werte_raw["Typ"].value_counts())
display(fms_werte_raw.groupby("Typ")["Wert"].describe())

Typ
4    17268
1     8634
5     8634
2     8634
Name: count, dtype: int64

,count,mean,std,min,25%,50%,75%,max
Typ,,,,,,,,
1,8634.0,18.209266,28.507100,0.0,0.000,0.0,38.8,96.800
2,8634.0,344070.385756,197234.933548,143450.3,184216.313,315364.3,502675.1,808361.063
4,17268.0,8705.683896,19185.153761,0.0,0.000,0.0,0.0,53598.000
5,8634.0,-32872.099259,65946.988797,-154875.0,0.000,0.0,0.0,43990.000


describe() ordnet die vier Codes ein. Typ 2 ist als einziges nie null (min 143.450) und liegt im selben Bereich wie der Kilometerstand (bis 808.361) — es ist der Kilometerstand. Typ 1 liegt zwischen 0 und rund 97 (Median 0), Typ 5 kann negativ werden (−154.875 bis 43.990, Median 0), Typ 4 kommt doppelt vor und liegt zwischen 0 und rund 53.600 (Median 0). Diese drei sind Tankfüllstand und Service-Intervall, also fahrzeugtechnische Signale ohne Bezug zu unseren Kennzahlen.

Entscheidend ist: Der einzige zielnahe Wert, der Kilometerstand, steht ohnehin schon in der GeoPosition jeder Meldung. Die FMS-Werte liefern für die Tourenanalyse damit nichts, was nicht bereits vorhanden wäre. Sie werden zwar separat als fms_werte_raw aufbewahrt, fließen aber in keine der vier Kennzahlen ein.

Bevor wir bereinigen, ein letzter Blick mit info(). Weil wir die Dateien als XML eingelesen haben, liegen alle Spalten als String vor - das ist der Ausgangspunkt für das Parsen im nächsten Kapitel. Außerdem sind die Zeitstempel unterschiedlich aufgebaut: Zeitstempel_GPS und Zeitstempel_Fahrzeug tragen einen Zeitzonen-Offset, Zeitstempel_Server nicht.

In [7]:
prc_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 33155 entries, 0 to 33154
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   msg_typ               33155 non-null  str  
 1   quelle_datei          33155 non-null  str  
 2   PositionsID           33155 non-null  str  
 3   Longitude             33155 non-null  str  
 4   Latitude              33155 non-null  str  
 5   Geschwindigkeit       33155 non-null  str  
 6   KMStand               33155 non-null  str  
 7   Zeitstempel_GPS       33155 non-null  str  
 8   Zeitstempel_Fahrzeug  33155 non-null  str  
 9   Zeitstempel_Server    33155 non-null  str  
 10  FahrzeugID            33155 non-null  str  
 11  Status                12984 non-null  str  
 12  TourID                1405 non-null   str  
 13  StationID             8299 non-null   str  
 14  SendposID             3280 non-null   str  
dtypes: str(15)
memory usage: 9.4 MB


## 3. Bereinigung und Vorbereitung
### 3.1 Zeitstempel parsen

Weil wir die Dateien als XML eingelesen haben, liegen alle Spalten als String vor und müssen in passende Typen überführt werden. Wir legen dafür mit prc eine Arbeitskopie an, prc_raw bleibt unverändert. Die zwei Zeitstempel mit Offset (Zeitstempel_GPS, Zeitstempel_Fahrzeug) parsen wir über UTC und rechnen sie auf lokale Zeit (Europe/Berlin) um, danach ohne Zeitzone. Die Sommerzeit-Umstellung Ende März wird so automatisch korrekt behandelt. Den Zeitstempel_Server parsen wir ohne UTC, da er keinen Offset trägt und bereits lokale Zeit ist. Damit liegen alle drei als lokale, tz-lose Zeit vor und sind mit den ebenfalls tz-losen Zeiten aus Soll und EMR vergleichbar. Längen- und Breitengrad, Geschwindigkeit und Kilometerstand werden zu Zahlen, Status als nullable Integer, weil er nur bei den Statusmeldungen belegt ist. Die ID-Felder bleiben bewusst Text.

In [8]:
prc = prc_raw.copy()

# Zeitstempel mit Offset über UTC auf lokale, tz-lose Zeit (Europe/Berlin) bringen
for col in ["Zeitstempel_GPS", "Zeitstempel_Fahrzeug"]:
    prc[col] = pd.to_datetime(prc[col], utc=True).dt.tz_convert("Europe/Berlin").dt.tz_localize(None)
prc["Zeitstempel_Server"] = pd.to_datetime(prc["Zeitstempel_Server"])  # ohne Offset -> ist bereits lokale Zeit

# GPS-Felder numerisch, Status als nullable Integer (nur bei Statusmeldungen belegt)
for col in ["Longitude", "Latitude", "Geschwindigkeit", "KMStand"]:
    prc[col] = pd.to_numeric(prc[col])
prc["Status"] = pd.to_numeric(prc["Status"]).astype("Int64")

prc.dtypes

msg_typ                            str
quelle_datei                       str
PositionsID                        str
Longitude                      float64
Latitude                       float64
Geschwindigkeit                  int64
KMStand                          int64
Zeitstempel_GPS         datetime64[us]
Zeitstempel_Fahrzeug    datetime64[us]
Zeitstempel_Server      datetime64[us]
FahrzeugID                         str
Status                           Int64
TourID                             str
StationID                          str
SendposID                          str
dtype: object

Nachdem wir die Zeitstempel geparsed haben, prüfen wir die drei Zeitbereiche der Zeitstempel_GPS/Fahrzeug/Server. Daraus ergibt sich, ob und wie wir auf den gemeinsamen Analysezeitraum für März filtern müssen. Außerdem entscheiden wir so, welcher Zeitstempel als fachlicher Meldungszeitpunkt für den Filter dient.

In [9]:
# Zeitbereich je Zeitstempel-Spalte vergleichen
for col in ["Zeitstempel_GPS", "Zeitstempel_Fahrzeug", "Zeitstempel_Server"]:
    print(f"{col}: {prc[col].min()}  bis  {prc[col].max()}")

# Meldungen je Kalendertag, mit Wochentag (erklärt die schwachen Tage)
tage = prc["Zeitstempel_Fahrzeug"].dt.date.value_counts().sort_index()
tage.index = [f"{t} ({pd.Timestamp(t).day_name()})" for t in tage.index]
tage

Zeitstempel_GPS: 2026-02-09 12:19:59  bis  2026-03-31 13:25:19
Zeitstempel_Fahrzeug: 2026-02-28 00:00:00  bis  2026-03-31 13:25:22
Zeitstempel_Server: 2026-02-28 00:00:43  bis  2026-03-31 13:26:09


2026-02-28 (Saturday)       18
2026-03-01 (Sunday)         24
2026-03-02 (Monday)       1452
2026-03-03 (Tuesday)      1232
2026-03-04 (Wednesday)    1340
2026-03-05 (Thursday)     1498
2026-03-06 (Friday)       1150
2026-03-07 (Saturday)       24
2026-03-08 (Sunday)         24
2026-03-09 (Monday)       1446
2026-03-10 (Tuesday)      1772
2026-03-11 (Wednesday)    1930
2026-03-12 (Thursday)     1722
2026-03-13 (Friday)       1388
2026-03-14 (Saturday)       33
2026-03-15 (Sunday)         21
2026-03-16 (Monday)       1590
2026-03-17 (Tuesday)      1778
2026-03-18 (Wednesday)    1476
2026-03-19 (Thursday)     1851
2026-03-20 (Friday)       1184
2026-03-21 (Saturday)       53
2026-03-22 (Sunday)         27
2026-03-23 (Monday)       1343
2026-03-24 (Tuesday)      1243
2026-03-25 (Wednesday)    1548
2026-03-26 (Thursday)     1846
2026-03-27 (Friday)        860
2026-03-28 (Saturday)       38
2026-03-29 (Sunday)         30
2026-03-30 (Monday)       1482
2026-03-31 (Tuesday)      1732
Name: co

### 3.2 Bezugszeitpunkt und Filter auf den Analysezeitraum

Die Zeitbereiche zeigen, dass die drei Stempel unterschiedlich weit zurückreichen: Zeitstempel_GPS beginnt schon am 09.02., Zeitstempel_Fahrzeug und Zeitstempel_Server erst am 28.02. Einzelne GPS-Stempel sind also veraltet (ein alter Fix, der mitgeschickt wurde) und taugen nicht als Meldungszeitpunkt. Wir verwenden Zeitstempel_Fahrzeug als fachlichen Meldungszeitpunkt — den Moment, in dem das Fahrzeug die Meldung erzeugt hat; Zeitstempel_Server markiert dagegen eher den Eingang im Backend.

Wie bei Soll und EMR begrenzen wir auf den März mit denselben Grenzen, angewandt auf Zeitstempel_Fahrzeug. Es fallen nur 18 Meldungen vom 28.02. weg — die PRC-Daten waren also schon weitgehend auf März beschränkt. In der Tagesverteilung tragen einzelne Tage nur 20 bis 50 statt 1.200 bis 1.900 Meldungen; das sind ausnahmslos Samstage und Sonntage, an denen der Betrieb ruht, und kein Datenproblem.

In [10]:
# Filter auf den gemeinsamen Analysezeitraum März, gleiche Grenzen wie Soll und EMR
vorher = len(prc)
prc = prc[(prc["Zeitstempel_Fahrzeug"] >= "2026-03-01") &
          (prc["Zeitstempel_Fahrzeug"] <  "2026-04-01")].copy()

print(f"Außerhalb März entfernt: {vorher - len(prc)} Zeilen | verbleibend: {len(prc)}")

Außerhalb März entfernt: 18 Zeilen | verbleibend: 33137


### 3.3 Fahrzeugkennungen vereinheitlichen

Die Übersicht zeigt 13 Kennungen und bestätigt die uneinheitlichen Schreibweisen aus der Vorschau — dasselbe Fahrzeug steht mal als Plö-TS853, mal als Plo-TS857, mal mit Leerzeichen wie OD-TS 8041. Darunter steckt auch Mafi mit 1.665 Meldungen, das wir wie in Soll und EMR auf Vorgabe von Frank entfernen, und zwar direkt vor der Normalisierung.

Die übrigen Kennungen vereinheitlichen wir mit derselben Funktion wie in den anderen Quellen: Großschreibung, Umlaute aufgelöst, Buchstaben- und Zifferngruppen mit Bindestrich. Ein Sonderfall bleibt: TS859 trägt kein Präfix. Laut den Fahrzeugstammdaten (Fahrzeugkennungen.xlsx) gehört diese Kennung zum Fahrzeug Plö-TS859, also PLO-TS-859. Unabhängig davon bestätigt ein Ausschluss dasselbe: Von zwölf Fahrzeugen sind elf zeichengleich mit dem Soll, übrig bleibt auf jeder Seite genau eines ohne Partner — im Soll PLO-TS-859, in PRC TS859. Abschließend benennen wir die Spalte in LKW_KENNZ um, passend zum gemeinsamen Schlüssel von Soll und EMR. Es bleiben zwölf Kennzeichen, dieselben wie in den anderen Quellen.

In [11]:
print("Eindeutige Fahrzeugkennungen:", prc["FahrzeugID"].nunique())
prc["FahrzeugID"].value_counts(dropna=False)

Eindeutige Fahrzeugkennungen: 13


FahrzeugID
PI-EN 444     4390
TS859         3794
Plö-TS853     3113
OD-TS 8041    3044
OD-TS 8048    2965
OD-TS 8050    2855
Plö-TS856     2647
WL-PL431      2579
OD-TS 8046    2351
Mafi          1665
OD-TS 8044    1638
PI-EN 900     1267
Plo-TS857      829
Name: count, dtype: int64

Die Übersicht zeigt 13 Kennungen und bestätigt die uneinheitlichen Schreibweisen aus der Vorschau. Darunter steckt auch Mafi mit 1.665 Meldungen. Dasselbe Fahrzeug haben wir in Soll und EMR auf Vorgabe von Frank entfernt und entfernen es hier ebenso, direkt vor der Normalisierung.

Die übrigen Kennungen vereinheitlichen wir mit derselben Funktion wie in den anderen Quellen: Großschreibung, Umlaute aufgelöst, Buchstaben- und Zifferngruppen mit Bindestrich. Ein Sonderfall bleibt: `TS859` trägt kein Präfix. Laut der Fahrzeugstammdaten (`Fahrzeugkennungen.xlsx`) gehört diese Kennung zum Fahrzeug `Plö-TS859`, also `PLO-TS-859`. Unabhängig davon bestätigt ein Ausschluss dasselbe: Von zwölf Fahrzeugen sind elf zeichengleich mit dem Soll, übrig bleibt auf jeder Seite genau eines — im Soll `PLO-TS-859`, in PRC `TS859`. Wir setzen den Alias daher gezielt und benennen die Spalte abschließend in `LKW_KENNZ` um.

In [12]:
def normalisiere_kennzeichen(wert):
    """Vereinheitlicht Fahrzeugkennungen wie in Soll und EMR: Großschreibung,
    Umlaute aufgelöst, Buchstaben-/Zifferngruppen mit Bindestrich getrennt."""
    if pd.isna(wert):
        return pd.NA
    wert = str(wert).upper().strip()
    wert = wert.replace("Ä", "A").replace("Ö", "O").replace("Ü", "U").replace("ß", "SS")
    return "-".join(re.findall(r"[A-Z]+|[0-9]+", wert))

# Mafi entfernen (Vorgabe Frank), vor der Normalisierung über den Rohnamen
vorher = len(prc)
prc = prc[prc["FahrzeugID"] != "Mafi"].copy()
print(f"Mafi-Zeilen entfernt: {vorher - len(prc)}")

# Normalisieren; TS859 (kein Präfix) gehört laut Stammdaten zu PLO-TS-859,
# unabhängig bestätigt durch Ausschluss: einziges Fahrzeug ohne Soll-Partner
prc["FahrzeugID"] = prc["FahrzeugID"].apply(normalisiere_kennzeichen).replace({"TS-859": "PLO-TS-859"})
prc = prc.rename(columns={"FahrzeugID": "LKW_KENNZ"})

prc["LKW_KENNZ"].value_counts().sort_index()

Mafi-Zeilen entfernt: 1665


LKW_KENNZ
OD-TS-8041    3044
OD-TS-8044    1638
OD-TS-8046    2351
OD-TS-8048    2965
OD-TS-8050    2855
PI-EN-444     4390
PI-EN-900     1267
PLO-TS-853    3113
PLO-TS-856    2647
PLO-TS-857     829
PLO-TS-859    3794
WL-PL-431     2579
Name: count, dtype: int64

## 4. Vertiefte Datenprüfung

Vor dem Export sehen wir uns die in der Übersicht notierten Auffälligkeiten genauer an: mögliche Dubletten, den Kilometerstand und die Plausibilität der Geodaten.

### 4.1 Dubletten

Eine echte Dublette ist hier eine über alle Inhaltsfelder identische Meldung. Zwei Felder dürfen deshalb nicht in den Vergleich. Die quelle_datei ist je Zeile verschieden, weil jede Meldung aus einer eigenen Datei stammt — würde man sie mitvergleichen, wäre keine Zeile je doppelt. Und der Zeitstempel_Server kann um Sekunden versetzt sein, wenn dieselbe vom Fahrzeug erzeugte Meldung über mehrere aufeinanderfolgende Importdateien ankommt, obwohl sie inhaltlich identisch ist. Die PositionsID taugt ebenfalls nicht als Zeilenschlüssel, da sich die in einer Datei gebündelten Meldungen denselben GPS-Eintrag und damit dieselbe PositionsID teilen. Wir definieren eine Dublette daher über alle Inhaltsfelder ohne quelle_datei und Zeitstempel_Server und sehen zuerst, welche Meldungstypen betroffen sind.

In [13]:
inhalt_cols = [c for c in prc.columns if c not in ["quelle_datei", "Zeitstempel_Server"]]

dubletten = (prc[prc.duplicated(subset=inhalt_cols, keep=False)]
             .sort_values(["PositionsID", "msg_typ", "quelle_datei"]))

print("Betroffene Zeilen insgesamt:", len(dubletten))
print("Verteilung nach Meldungstyp:")
print(dubletten["msg_typ"].value_counts(dropna=False))
display(dubletten[["msg_typ", "quelle_datei", "PositionsID", "Zeitstempel_Fahrzeug",
                   "Zeitstempel_Server", "Status", "StationID"]].head(12))

Betroffene Zeilen insgesamt: 3156
Verteilung nach Meldungstyp:
msg_typ
FMSStatus         2986
Stationsstatus     170
Name: count, dtype: int64


,msg_typ,quelle_datei,PositionsID,Zeitstempel_Fahrzeug,Zeitstempel_Server,Status,StationID
143,FMSStatus,msg_20260302054940_1063.imp.20260302055135386.prc,16056058618,2026-03-02 05:49:36,2026-03-02 05:49:40,<NA>,NaN
145,FMSStatus,msg_20260302054940_1064.imp.20260302055135419.prc,16056058618,2026-03-02 05:49:36,2026-03-02 05:49:40,<NA>,NaN
157,FMSStatus,msg_20260302055042_1070.imp.20260302055135882.prc,16056063396,2026-03-02 05:50:17,2026-03-02 05:50:42,<NA>,NaN
159,FMSStatus,msg_20260302055042_1071.imp.20260302055135966.prc,16056063396,2026-03-02 05:50:17,2026-03-02 05:50:42,<NA>,NaN
161,FMSStatus,msg_20260302055042_1072.imp.20260302055135999.prc,16056063396,2026-03-02 05:50:17,2026-03-02 05:50:42,<NA>,NaN
182,FMSStatus,msg_20260302060251_1087.imp.20260302060645331.prc,16056162705,2026-03-02 05:59:12,2026-03-02 06:02:51,<NA>,NaN
184,FMSStatus,msg_20260302060251_1088.imp.20260302060645445.prc,16056162705,2026-03-02 05:59:12,2026-03-02 06:02:51,<NA>,NaN
186,FMSStatus,msg_20260302060251_1089.imp.20260302060645467.prc,16056162705,2026-03-02 05:59:12,2026-03-02 06:02:51,<NA>,NaN
205,FMSStatus,msg_20260302060626_1101.imp.20260302060646144.prc,16056184183,2026-03-02 06:05:48,2026-03-02 06:06:26,<NA>,NaN
207,FMSStatus,msg_20260302060626_1102.imp.20260302060646214.prc,16056184183,2026-03-02 06:05:48,2026-03-02 06:06:26,<NA>,NaN


Die betroffenen Zeilen verteilen sich auf die zwei Meldungstypen FMSStatus mit 2.986 und Stationsstatus mit 170. An den Quelldateinamen ist erkennbar, dass dieselbe Meldung in mehreren
aufeinanderfolgenden Dateien desselben Import-Vorgangs ankam (fortlaufende Nummern und Importzeit im Millisekundenabstand). Manche Meldungen kamen sogar zwei- bis viermal. Beim Entfernen fallen
daher weniger Zeilen weg, als hier betroffen sind, weil wir je Meldung eine behalten.

Besonders relevant sind die 170 Stationsstatus, da ein doppelt erfasstes Stationsereignis (z. B. eine Ankunft) später eine Standzeit doppelt zählen würde.

In [14]:
vorher = len(prc)
prc = prc.drop_duplicates(subset=inhalt_cols).copy()

print(f"Zeilen vorher:           {vorher}")
print(f"Inhaltsgleiche entfernt: {vorher - len(prc)}")
print(f"Zeilen jetzt:            {len(prc)}")

Zeilen vorher:           31472
Inhaltsgleiche entfernt: 2043
Zeilen jetzt:            29429


### 4.2 Übersicht: fehlende Werte, eindeutige Werte und Statistik

Da wir zu Beginn nur .info() ausgeführt haben, ergänzen wir hier die Übersicht analog zu Soll und EMR. Also Datentypen, Anzahl und Anteil fehlender Werte sowie die Zahl eindeutiger Werte und dazu die Statistik der numerischen Felder wie dem Kilometerstand und den Geodaten.

In [15]:
prc_overview = pd.DataFrame({
    "datentyp": prc.dtypes.astype(str),
    "fehlende_werte": prc.isna().sum(),
    "fehlende_werte_%": (prc.isna().mean() * 100).round(2),
    "eindeutige_werte": prc.nunique(dropna=True),
})
display(prc_overview)

print("Statistik der numerischen Felder")
display(prc[["Longitude", "Latitude", "Geschwindigkeit", "KMStand"]].describe())

,datentyp,fehlende_werte,fehlende_werte_%,eindeutige_werte
msg_typ,str,0,0.00,6
quelle_datei,str,0,0.00,22753
PositionsID,str,0,0.00,19831
Longitude,float64,0,0.00,8462
Latitude,float64,0,0.00,7635
Geschwindigkeit,int64,0,0.00,118
KMStand,int64,0,0.00,5868
Zeitstempel_GPS,datetime64[us],0,0.00,16459
Zeitstempel_Fahrzeug,datetime64[us],0,0.00,16108
Zeitstempel_Server,datetime64[us],0,0.00,9876


Statistik der numerischen Felder


,Longitude,Latitude,Geschwindigkeit,KMStand
count,29429.000000,29429.000000,29429.000000,29429.000000
mean,9.959395,53.567366,13.517653,284365.169153
std,0.344909,0.190485,22.629284,223999.814903
min,8.165830,51.298860,0.000000,0.000000
25%,9.956840,53.522630,0.000000,179093.000000
50%,9.985700,53.523530,0.000000,195984.000000
75%,9.992030,53.627730,23.000000,462479.000000
max,13.647510,54.781390,128.000000,808364.000000


Die fehlenden Werte folgen durchgängig der typgebundenen Belegung. Status, TourID, StationID und SendposID sind nur bei den Meldungstypen gefüllt, die sie tragen, und entsprechend hoch ist der Fehlanteil über alle Zeilen. Die immer belegten Felder (GPS, Zeitstempel, Fahrzeug) haben keine Lücken. Die Statistik zeigt allerdings einen Kilometerstand mit min = 0 bei einem Median um 196.000 und wir überprüfen außerdem die Spannweite der Koordinaten.

### 4.3 Kilometerstand

Wir sehen uns an, bei welchen Meldungstypen der Kilometerstand = 0 auftritt. Daraus ergibt sich, ob es ein durchgehendes Merkmal einzelner Typen ist oder quer durch die Daten läuft.

In [16]:
n_null = (prc["KMStand"] == 0).sum()
print(f"KMStand == 0: {n_null}  ({n_null / len(prc) * 100:.1f}%)")
print("0-Werte nach Meldungstyp:")
print(prc[prc["KMStand"] == 0]["msg_typ"].value_counts())
print("KMStand je Meldungstyp (min / median / max):")
print(prc.groupby("msg_typ")["KMStand"].agg(["min", "median", "max"]))

KMStand == 0: 5533  (18.8%)
0-Werte nach Meldungstyp:
msg_typ
Position          2565
Stationsstatus    1755
Sendposstatus     1002
Tourstatus         211
Name: count, dtype: int64
KMStand je Meldungstyp (min / median / max):
                   min    median     max
msg_typ                                 
Ereignis        461908  634466.0  807024
FMSStatus       143450  315344.0  808361
Position             0  196324.0  808364
Sendposstatus        0  194994.0  808360
Stationsstatus       0  186750.0  808361
Tourstatus           0  194994.0  807996


Es zeigt sich, dass die 0-Werte nicht an einem einzelnen Meldungstyp hängen. Sie treten bei Position, Stationsstatus, Sendposstatus und Tourstatus auf, während dieselben Typen sonst plausible Kilometerstände tragen (Median rund 187.000–196.000). Nur FMSStatus und Ereignis sind durchgehend echt. Die 0er-Kilometerstände sind also ein quer durch die Typen laufender Teilbestand. Wir schauen uns an, ob sie sich auf bestimmte Fahrzeuge konzentrieren.

In [17]:
print("KMStand=0 Anteil je Fahrzeug (%):")
print(prc.groupby("LKW_KENNZ")["KMStand"]
        .apply(lambda s: (s == 0).mean() * 100).round(1).sort_values(ascending=False))

KMStand=0 Anteil je Fahrzeug (%):
LKW_KENNZ
OD-TS-8044    100.0
PLO-TS-853    100.0
PLO-TS-857    100.0
OD-TS-8041      0.0
OD-TS-8048      0.0
OD-TS-8046      0.0
PI-EN-444       0.0
OD-TS-8050      0.0
PI-EN-900       0.0
PLO-TS-856      0.0
PLO-TS-859      0.0
WL-PL-431       0.0
Name: KMStand, dtype: float64


Die 0-Werte sind fahrzeugbedingt. Es sind drei Fahrzeuge (OD-TS-8044, PLO-TS-853, PLO-TS-857), die durchgehend einen Kilometerstand von 0 melden, während alle anderen durchgehend echte Werte tragen. Es ist also keine situative Lücke, sondern ein Merkmal genau dieser drei Fahrzeuge. Weil Frank erwähnt hat, dass nur ein Teil der Flotte einen Fahrtenschreiber besitzt, prüfen wir im nächsten Schritt, ob dieselben drei Fahrzeuge auch beim FMSStatus fehlen — denn der Kilometerstand stammt aus demselben Fahrzeugbus.

GPS-Spur, Zeiten und Status dieser Fahrzeuge sind vollständig, nur der Kilometerstand fehlt, daher behalten wir alle Zeilen. Für die spätere Kilometer-Abweichung heißt das, dass für diese drei Fahrzeuge aus PRC kein Kilometerstand vorliegt.

In [18]:
print("FMSStatus-Meldungen je Fahrzeug:")
print(prc.loc[prc["msg_typ"] == "FMSStatus", "LKW_KENNZ"].value_counts().sort_index())

FMSStatus-Meldungen je Fahrzeug:
LKW_KENNZ
OD-TS-8041     687
OD-TS-8046     465
OD-TS-8048     665
OD-TS-8050     694
PI-EN-444     1477
PI-EN-900      327
PLO-TS-856     549
PLO-TS-859    1095
WL-PL-431      717
Name: count, dtype: int64


Die Prüfung bestätigt es: Nur neun Fahrzeuge senden überhaupt FMSStatus, und genau die drei 0-km-Fahrzeuge (OD-TS-8044, PLO-TS-853, PLO-TS-857) fehlen dort. Beide Merkmale, kein Kilometerstand und kein FMSStatus, treffen dieselben drei Fahrzeuge. Damit ist belegt, dass es die drei Fahrzeuge ohne Fahrtenschreiber sind — eine Gegenprüfung in den EMR-Daten und Franks Auskunft bestätigen dieselbe Dreiergruppe. Für die Kilometer-Lücke bringt der FMS-Block also nichts, weil er genau bei den betroffenen Fahrzeugen fehlt. Wir verwerfen ihn dennoch nicht, da sich die Bedeutung der übrigen Codes womöglich noch klären lässt, und exportieren fms_werte separat, auf dieselbe Basis wie prc gebracht (März, ohne Mafi, normalisierte Kennzeichen).

In [19]:
# Die FMSStatus-Meldungen, die nach der Dubletten-Bereinigung übrig sind, 
# identifiziert über ihre PositionsID. März-Filter und Mafi-Entfernung werden automatisch geerbt.
fms_meldungen = prc.loc[prc["msg_typ"] == "FMSStatus", ["PositionsID", "quelle_datei", "LKW_KENNZ"]]

fms_werte = fms_werte_raw.merge(fms_meldungen, on=["PositionsID", "quelle_datei"], how="inner")
fms_werte["Wert"] = pd.to_numeric(fms_werte["Wert"])
fms_werte = fms_werte.drop(columns="quelle_datei")

print("FMS-Werte:", len(fms_werte))
print("Zugeordnete Meldungen:", fms_werte["PositionsID"].nunique(), "von", len(fms_meldungen), "FMSStatus")
print(fms_werte["Typ"].value_counts().sort_index())

FMS-Werte: 33380
Zugeordnete Meldungen: 6676 von 6676 FMSStatus
Typ
1     6676
2     6676
4    13352
5     6676
Name: count, dtype: int64


### 4.4 Geodaten

Die Statistik zeigt Längengrad 8,17–13,65 und Breitengrad 51,30–54,78, alles innerhalb Deutschlands und ohne 0-Koordinaten. Das allein schließt einzelne fehlerhafte GPS-Sprünge aber nicht aus. Wir prüfen daher zweistufig: erst die vier Extrempunkte (liegen die Ränder an realen Orten?), dann die Verteilung aller Meldungen außerhalb des Großraums Hamburg über die Fahrzeuge (viele Meldungen = echte Fahrt, vereinzelte = Ausreißer).

In [20]:
print("Extrempunkte (min/max Längen- und Breitengrad):")
extrem = pd.concat([
    prc.nsmallest(1, "Longitude"), prc.nlargest(1, "Longitude"),
    prc.nsmallest(1, "Latitude"),  prc.nlargest(1, "Latitude"),
])
display(extrem[["msg_typ", "LKW_KENNZ", "Zeitstempel_Fahrzeug",
                "Longitude", "Latitude", "Geschwindigkeit"]])

# Meldungen außerhalb des Großraums Hamburg: echte Fahrten oder Einzelausreißer?
fern = prc[(prc["Longitude"] > 11) | (prc["Longitude"] < 9) |
           (prc["Latitude"] < 52.5) | (prc["Latitude"] > 54)]
print(f"Meldungen außerhalb des Großraums Hamburg: {len(fern)}  ({len(fern) / len(prc) * 100:.1f}%)")
print("davon je Fahrzeug:")
print(fern["LKW_KENNZ"].value_counts())

Extrempunkte (min/max Längen- und Breitengrad):


,msg_typ,LKW_KENNZ,Zeitstempel_Fahrzeug,Longitude,Latitude,Geschwindigkeit
2192,Position,OD-TS-8050,2026-03-03 10:26:18,8.16583,53.28381,11
16650,Stationsstatus,OD-TS-8050,2026-03-16 19:27:11,13.64751,54.14740,0
32928,Position,WL-PL-431,2026-03-31 11:50:06,11.99148,51.29886,26
23632,Stationsstatus,PLO-TS-856,2026-03-23 09:06:43,8.83709,54.78139,0


Meldungen außerhalb des Großraums Hamburg: 1162  (3.9%)
davon je Fahrzeug:
LKW_KENNZ
OD-TS-8048    279
PLO-TS-856    264
OD-TS-8050    257
PLO-TS-853    173
WL-PL-431      55
OD-TS-8041     50
OD-TS-8044     50
PLO-TS-857     34
Name: count, dtype: int64


Die Extrempunkte sind reale Fahrzeuge zu plausiblen Zeiten an stimmigen Orten. Die 1.162 Meldungen außerhalb des Großraums Hamburg (3,9 %) verteilen sich über acht Fahrzeuge mit jeweils dutzenden bis hunderten Meldungen, also echte Langstreckentouren, keine Einzelausreißer. Die Geodaten sind damit plausibel und wir nehmen keine weiteren Änderung vor.

## 5. Export

Wir schreiben zwei Tabellen nach data/processed/: die bereinigten PRC-Meldungen (prc) und die separat aufbereiteten FMS-Werte (fms_werte). Beide je als CSV und Pickle. Das Pickle bewahrt die Datentypen, besonders die geparsten Zeitstempel und den nullable-Integer-Status, sodass das Analyse-Notebook ohne erneute Typumwandlung weiterarbeiten kann; die CSV dient der Sichtbarkeit und als Beleg. Dass PRC als einzige Quelle zwei Tabellen exportiert, liegt an den FMS-Werten, die in einer eigenen Eins-zu-viele-Beziehung zur Meldung stehen und deshalb getrennt gehalten werden.

In [21]:
export = Path.cwd().parent / "data" / "processed"
export.mkdir(parents=True, exist_ok=True)

# Zwei Tabellen, je zwei Formate: Pickle erhält die Datentypen (Zeitstempel, nullable Status)
# fürs Analyse-Notebook, CSV ist die lesbare Beleg-Kopie
for name, df in [("istdaten_prc_clean", prc), ("istdaten_prc_fmswerte", fms_werte)]:
    df.to_csv(export / f"{name}.csv", index=False)
    df.to_pickle(export / f"{name}.pkl")
    print(f"{name}: {len(df)} Zeilen | CSV + Pickle")

istdaten_prc_clean: 29429 Zeilen | CSV + Pickle
istdaten_prc_fmswerte: 33380 Zeilen | CSV + Pickle
